# Лабораторная работа 1

Тема: численное дифференцирование и методы оптимизации первого порядка.

В ноутбуке используются C++23-биндинги `optlib`: конечные разности, multidual autograd, функция Розенброка и оптимизаторы GD, HeavyBall, Nesterov, Adam, RMSProp, Adagrad.

In [ ]:
from __future__ import annotations

import time

import matplotlib.pyplot as plt
import numpy as np

import optlib

try:
    from scipy.optimize import minimize
except ModuleNotFoundError:
    minimize = None

print(optlib.__version__)

## Розенброк и траектории

Запускаем все методы из одной стартовой точки. Логирование траектории включено, чтобы построить путь по контурной карте.

In [ ]:
METHODS = {
    "gradient_descent": {"learning_rate": 1e-3},
    "heavy_ball": {"learning_rate": 8e-4, "momentum": 0.8},
    "nesterov": {"learning_rate": 8e-4, "momentum": 0.8},
    "adam": {"learning_rate": 2e-2},
    "rmsprop": {"learning_rate": 2e-4, "beta2": 0.999},
    "adagrad": {"learning_rate": 5e-2},
}

x0 = np.array([-1.2, 1.0])
results = {}

for method, params in METHODS.items():
    started_at = time.perf_counter()
    result = optlib.MinimizeRosenbrock(
        x0,
        method=method,
        max_iter=20_000,
        gradient_tolerance=1e-3,
        step_tolerance=0.0,
        function_tolerance=0.0,
        log_trajectory=True,
        **params,
    )
    result["wall_ms"] = (time.perf_counter() - started_at) * 1000.0
    results[method] = result

for method, result in results.items():
    print(
        f"{method:16s} f={result['value']:.3e} "
        f"||g||={result['gradient_norm']:.3e} "
        f"iter={result['iterations']:5d} x={result['x']}"
    )

In [ ]:
grid_x = np.linspace(-2.0, 2.0, 300)
grid_y = np.linspace(-1.0, 3.0, 300)
mesh_x, mesh_y = np.meshgrid(grid_x, grid_y)
mesh_z = 100.0 * (mesh_y - mesh_x**2) ** 2 + (1.0 - mesh_x) ** 2

plt.figure(figsize=(9, 7))
levels = np.logspace(-1, 3, 18)
plt.contour(mesh_x, mesh_y, mesh_z, levels=levels, cmap="viridis")

for method, result in results.items():
    path = result["trajectory"]["x"]
    stride = max(1, path.shape[0] // 400)
    shown_path = path[::stride]
    plt.plot(shown_path[:, 0], shown_path[:, 1], marker=".", ms=2, label=method)

plt.scatter([1.0], [1.0], c="red", marker="*", s=120, label="minimum")
plt.title("Траектории на функции Розенброка")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

## Кривые сходимости

Сравниваем значение функции и норму градиента по итерациям. Для Розенброка важна не только высота, но и движение вдоль узкой долины.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for method, result in results.items():
    trajectory = result["trajectory"]
    axes[0].semilogy(trajectory["f"], label=method)
    axes[1].semilogy(trajectory["grad_norm"], label=method)

axes[0].set_title("Значение функции")
axes[0].set_xlabel("Итерация")
axes[0].set_ylabel("f(x)")
axes[1].set_title("Норма градиента")
axes[1].set_xlabel("Итерация")
axes[1].set_ylabel("||grad f(x)||")

for axis in axes:
    axis.grid(True, which="both", alpha=0.25)
    axis.legend()

plt.show()

## Многомерная проверка

Розенброк реализован для произвольной размерности. Здесь проверяем, что тот же Adam-интерфейс работает для нескольких `n` без изменения кода оптимизатора.

In [ ]:
for dimension in [2, 5, 10]:
    start = np.empty(dimension)
    start[0::2] = -1.2
    start[1::2] = 1.0
    result = optlib.MinimizeRosenbrock(
        start,
        method="adam",
        learning_rate=2e-2,
        max_iter=30_000,
        gradient_tolerance=1e-3,
        step_tolerance=0.0,
        function_tolerance=0.0,
        log_trajectory=False,
    )
    print(
        f"n={dimension:2d} f={result['value']:.3e} "
        f"||g||={result['gradient_norm']:.3e} iter={result['iterations']}"
    )

## Точность численного дифференцирования

Сравниваем три схемы конечных разностей с аналитическим градиентом. При слишком маленьком `h` ошибка округления начинает доминировать.

In [ ]:
x_ref = np.array([-1.2, 1.0, 0.8])
true_gradient = optlib.RosenbrockGradient(x_ref)
steps = np.logspace(-12, -1, 12)
schemes = ["forward", "central", "five-point"]
errors = {scheme: [] for scheme in schemes}

for scheme in schemes:
    for step in steps:
        gradient = optlib.RosenbrockNumericalGradient(x_ref, scheme, step)
        errors[scheme].append(np.linalg.norm(gradient - true_gradient))

plt.figure(figsize=(8, 5))
for scheme in schemes:
    plt.loglog(steps, errors[scheme], marker="o", label=scheme)

reference_step = steps[4:]
reference_scale = errors["forward"][4]
plt.loglog(
    reference_step,
    reference_scale * (reference_step / reference_step[0]),
    "--",
    color="gray",
    alpha=0.5,
    label="O(h)",
)
plt.loglog(
    reference_step,
    reference_scale * (reference_step / reference_step[0]) ** 2,
    ":",
    color="gray",
    alpha=0.5,
    label="O(h^2)",
)

plt.title("Ошибка градиента в зависимости от шага h")
plt.xlabel("h")
plt.ylabel("||g_numeric - g_true||")
plt.grid(True, which="both", alpha=0.25)
plt.legend()
plt.show()

## Autograd против конечных разностей

Forward-mode multidual дает градиент без подбора шага. Ниже — короткое сравнение точности и времени вызова.

In [ ]:
def median_time_ms(callable_object, repeats: int = 200) -> float:
    samples = []
    for _ in range(repeats):
        started_at = time.perf_counter()
        callable_object()
        samples.append((time.perf_counter() - started_at) * 1000.0)
    return float(np.median(samples))


autograd_gradient = optlib.RosenbrockAutogradGradient(x_ref)
central_gradient = optlib.RosenbrockNumericalGradient(x_ref, "central")
five_point_gradient = optlib.RosenbrockNumericalGradient(x_ref, "five-point")

print("autograd error", np.linalg.norm(autograd_gradient - true_gradient))
print("central error ", np.linalg.norm(central_gradient - true_gradient))
print("five-point error", np.linalg.norm(five_point_gradient - true_gradient))

benchmarks = {
    "analytic": lambda: optlib.RosenbrockGradient(x_ref),
    "autograd": lambda: optlib.RosenbrockAutogradGradient(x_ref),
    "central": lambda: optlib.RosenbrockNumericalGradient(x_ref, "central"),
    "five-point": lambda: optlib.RosenbrockNumericalGradient(x_ref, "five-point"),
}

for name, callable_object in benchmarks.items():
    print(f"{name:10s}: {median_time_ms(callable_object):.4f} ms")

## Сравнение со SciPy

SciPy нужен только для внешнего эталона. Если пакет не установлен, выполните `uv sync --extra experiments --extra dev`.

In [ ]:
if minimize is None:
    print("SciPy is not installed; skip external baseline.")
else:
    scipy_started_at = time.perf_counter()
    scipy_result = minimize(
        lambda values: optlib.RosenbrockValue(np.asarray(values, dtype=np.float64)),
        x0,
        jac=lambda values: optlib.RosenbrockGradient(np.asarray(values, dtype=np.float64)),
        method="BFGS",
        options={"gtol": 1e-6, "maxiter": 20_000},
    )
    scipy_wall_ms = (time.perf_counter() - scipy_started_at) * 1000.0

    print("SciPy BFGS")
    print("success", scipy_result.success)
    print("f", scipy_result.fun)
    print("x", scipy_result.x)
    print("iterations", scipy_result.nit)
    print("wall_ms", scipy_wall_ms)

    print("\noptlib Adam")
    print("f", results["adam"]["value"])
    print("x", results["adam"]["x"])
    print("iterations", results["adam"]["iterations"])
    print("wall_ms", results["adam"]["wall_ms"])